# Road Damage Inspection System

**Team:** Member 1 — Austin Wang · Member 2 — Kevin Fan · Member 3 — RJ Xia

## 8:17 a.m. — one road image, three urgent questions

> **What broke? Where is it? How much road surface is affected?**

<p align="center"><img src="artifacts/final/marked_pothole_hook.jpg" alt="A road pothole circled in white paint for repair" width="920"></p>

**Opening line:** An inspection vehicle may see this road only once, but a city may have thousands of frames waiting. Our goal is to turn each frame into visible evidence for an inspector—not an autonomous repair order.

**Project in one sentence:** road pixels → boxes + masks → explained priority → human judgment.


Good morning. Imagine that an inspection vehicle passes this damaged lane at 8:17 in the morning. The camera may see the road only once, while the city may have thousands of images waiting for review. From each image, an inspector needs three practical answers: what kind of damage is present, where it is located, and how much surface is affected. Our project combines detection, segmentation, and a transparent priority rule to organize that evidence. The system does not authorize repairs; it helps a human reviewer decide which images deserve attention first. During the presentation, we will follow that evidence from raw data to results, failures, and deployment.

# One image, two specialists, one reviewable decision

<p align="center"><img src="artifacts/presentation/project_workflow.svg" alt="End-to-end project workflow from data and road image to two models, evidence fusion, and human review" width="1100"></p>

**Takeaway:** YOLO answers **what and where**; SegFormer answers **how much area**. Their outputs are fused only to explain and rank cases for human review.


This diagram summarizes our complete workflow. The same road image goes to two specialists. YOLO11s is the spotter: it identifies the damage type and draws a bounding box. SegFormer-B0 is the measurer: it produces a pixel-level pothole mask and estimates the affected footprint. We then display both forms of evidence, together with confidence and simple reasons, in one review interface. The green arrows represent training sources; the blue path represents inference on one new image. The important boundary is on the right: the models suggest and explain, but the inspector makes the final decision.

# A bounding box and a mask answer different questions

| Evidence | What the audience can see | What it cannot prove alone |
|---|---|---|
| **Detection box** | Damage class, approximate location, confidence | Exact damaged surface or depth |
| **Segmentation mask** | Pixel-level pothole footprint and area | Crack category or engineering severity |
| **Combined view** | Type, location, footprint, confidence, and reasons | A certified maintenance decision |

**Takeaway:** detection and segmentation are complementary measurements; neither output should be mistaken for a complete road-condition assessment.

A bounding box is intentionally coarse: it tells us where a detector believes an object lies, but it includes undamaged pixels inside the rectangle. A segmentation mask answers a finer geometric question by classifying individual pixels, yet our mask model is binary and only describes potholes. Combining the two outputs gives an inspector more useful evidence, but it does not create new ground truth. The system still cannot infer pothole depth, traffic risk, repair cost, or structural condition from a single image.

# What do D00, D10, D20, and D40 mean?

RDD2022 uses four detection codes, each paired with a bounding box:

| Code | Meaning | Visual cue |
|---|---|---|
| **D00** | Longitudinal crack | Runs along the road |
| **D10** | Transverse crack | Crosses the road |
| **D20** | Alligator crack | Connected web of cracks |
| **D40** | Pothole | Local broken/depressed pavement |

<p align="center"><img src="artifacts/member1/EDA_Figures/09_damage_class_vocabulary.png" alt="RDD2022 examples of D00, D10, D20, and D40 with bounding boxes" width="960"></p>

**Takeaway:** the codes identify **damage type, not severity**—D40 is not automatically “level 40” damage.


Before discussing models, we need a shared vocabulary. RDD2022 labels four damage types. D00 is a longitudinal crack running along the road. D10 is a transverse crack crossing it. D20 is an alligator pattern, where connected cracks form a web. D40 is a pothole, meaning a localized broken or depressed area. The examples show that visually similar road patterns can still belong to different categories. These are category codes, not severity scores. A D40 label does not automatically mean the most dangerous case; severity still depends on size, confidence, context, and human inspection.

# Two public datasets, two complementary label types

| Data source | What it contains | Labels | Role in our project |
|---|---:|---|---|
| **RDD2022** | 47,420 images; 38,385 publicly labeled | 55,006 valid target boxes across six countries / seven capture domains | Four-class object detection |
| **Pothole Mix** | 4,340 image-mask pairs from six component sources | Pixel masks; official 3,340 / 496 / 504 split | Binary pothole segmentation |

<p align="center"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned Pothole Mix road images and segmentation masks" width="900"></p>

**Takeaway:** these datasets are related road imagery but solve different cognitive problems: RDD2022 supplies **boxes and types**, while Pothole Mix supplies **pixel-level pothole shape**.


We use two public road datasets because detection and segmentation require different labels. RDD2022 contains 47,420 images collected across six countries and seven capture domains. Of those, 38,385 images have public bounding-box annotations, giving us the location and type of four road-damage classes. Pothole Mix is smaller, with 4,340 image-and-mask pairs, but its labels trace potholes at pixel level. Keeping these label types separate prevents us from treating a box as if it were a precise surface mask. In short, RDD2022 teaches the detector what and where, while Pothole Mix teaches the segmenter the exact pothole shape.

# Not every official image is supervised training data

<p align="center"><img src="artifacts/member1/EDA_Figures/00_all_annotation_code_counts.png" alt="All annotation codes found in the official RDD2022 XML files" width="940"></p>

| RDD2022 record type | Count | Project use |
|---|---:|---|
| Official images inventoried | 47,420 | File and image-level audit |
| Images with public XML | 38,385 | Supervised detection pool |
| Official-test images without public XML | 9,035 | Image-level EDA only |
| Valid target-negative images | 14,618 | Retained as background examples |

**Takeaway:** missing public ground truth is not the same as “no damage,” and non-target codes are audited rather than silently remapped.

The official release contains more than the four classes used in our project. We found 65,712 raw XML boxes, but only D00, D10, D20, and D40 belong to the agreed target vocabulary. The other 10,705 annotations include codes such as D44, D50, repair, and region-specific labels. We preserve them in the audit trail but do not force them into a target class. We also keep the 9,035 official-test images separate because no public XML means that supervised accuracy cannot be measured on them.

# Member 1: make the data trustworthy before modeling

| Full-data audit | Verified result | Decision |
|---|---:|---|
| Official RDD2022 images decoded | 47,420 / 47,420 | No corrupt image removal |
| Publicly labeled images retained | 38,385 | Use the complete labeled pool |
| Raw XML boxes / target boxes | 65,712 / 55,007 | Audit 10,705 non-target codes; do not remap them |
| Exportable target boxes | 55,006 | Exclude one degenerate D20 box |
| Exact / exact+near duplicate rows | 4 / 3,256 | Retain but group-lock to one split |
| Valid target negatives | 14,618 images | Retain to teach background |

**Method chain:** Pascal VOC XML → image/box tables → integrity and duplicate audit → group-aware split → synchronized YOLO + COCO exports.

**Takeaway:** Member 1’s main product is not just EDA—it is a fixed, leakage-aware data contract shared by both detectors.


My role was to turn the raw RDD2022 release into a trustworthy modeling contract. I decoded all 47,420 images and found no corrupt files. I retained every publicly labeled image, audited both target and non-target XML codes, and removed only one degenerate target box from export. I also detected exact and near duplicates. Instead of deleting useful scenes, I group-locked related images so that they could not leak across train, validation, and test. Finally, I exported synchronized YOLO and COCO labels so later experiments used the same verified records. This lets every modeling result trace back to a stable image, split, and label version.

# Three data-quality choices protect the experiment

| Risk | Tempting shortcut | Our decision |
|---|---|---|
| Exact and near duplicates | Randomly split every file | Lock each duplicate group to one split |
| Images with no target boxes | Delete them as “empty” | Keep valid negatives to teach background |
| One degenerate D20 box | Export it anyway | Exclude the invalid box and document it |

**Takeaway:** data cleaning is not simply deletion; each decision must preserve useful information while preventing leakage or malformed supervision.

These choices matter because a clean-looking dataset can still produce misleading results. If near-duplicate frames appear in both training and test, the model may recognize a scene instead of generalizing to new roads. If all target-negative images are removed, the detector sees too little normal pavement and may generate more false alarms. If a zero-area box is exported, the training target is mathematically invalid. Group-locking, retaining negatives, and excluding one malformed box give us a stronger evaluation without discarding the full labeled pool.

# EDA prediction: small and rare damage will be the hard case

<table><tr>
<td width="50%"><img src="artifacts/member1/EDA_Figures/01_counts_country_and_class.png" alt="Image counts by country and target boxes by class" width="100%"></td>
<td width="50%"><img src="artifacts/member1/EDA_Figures/04_box_area_violin_and_center_heatmap.png" alt="Object scale by class and object center heatmap" width="100%"></td>
</tr></table>

- Japan contributes 10,506 labeled images versus 2,829 from Czech.
- D00 supplies 47.3% of target boxes; D40 supplies only 11.9%.
- The median target occupies 1.43% of the image; D40 has the smallest class median (0.54%).

**Takeaway:** imbalance and tiny targets mean that overall mAP is insufficient—we need per-class recall, especially for potholes.


The EDA gave us a prediction before training: small and rare damage would be hardest. The dataset is uneven across both geography and class. Japan contributes far more labeled images than Czech, and D00 supplies almost half of all target boxes, while D40 potholes supply only about twelve percent. Scale is an even stronger warning. The median target occupies only 1.43 percent of its image, and the median D40 box occupies just 0.54 percent. The class chart and box-area plot make these two risks visible at a glance. That is why later results emphasize per-class recall and small-object misses instead of reporting only one overall mAP number.

# The road domain changes beyond the damage label

<p align="center"><img src="artifacts/member1/EDA_Figures/08_visual_outliers.png" alt="RDD2022 examples with unusual lighting, blur, framing, snow, and camera viewpoints" width="980"></p>

- Drone frames may contain black borders and a near-vertical viewpoint.
- Underpasses, glare, snow, shadows, and windshield blur change local contrast.
- Country and capture device change resolution, road texture, markings, and scale.

**Takeaway:** a model must recognize damage across acquisition conditions, not only memorize the visual style of one camera or country.

These examples are valid images, not corrupt files, so simply deleting them would make the benchmark easier but less realistic. They show why standard resizing is necessary for model input but cannot erase domain differences. A crack visible in a close motorbike image may occupy only a few pixels in a drone or distant dashboard frame. This visual diversity motivates the held-out-country experiment and explains why performance should be reviewed by object size, geography, brightness, and blur rather than by one global metric alone.

# From EDA to a fair evaluation design

<p align="center"><img src="artifacts/member1/EDA_Figures/05_resolution_brightness_contrast_blur.png" alt="Resolution, brightness, contrast, and blur distributions by country" width="930"></p>

| Leakage-safe split | Images |
|---|---:|
| Train | 26,888 |
| Validation | 5,714 |
| Test | 5,783 |
| Held-out-US auxiliary test | 4,805 |

Norway’s median resolution is 8.22 MP while most domains are roughly 0.26–0.52 MP; brightness and capture viewpoint also vary by country. Duplicate groups never cross the primary split.

**Takeaway:** a random held-out test measures average performance; the separate held-out-US test asks whether the detector travels to a new geography.


The image statistics also vary strongly by capture domain. Norway has much higher-resolution images, while viewpoint, brightness, contrast, and blur shift across countries. We therefore created a duplicate-safe primary split for average performance and a separate held-out-US experiment for geographic transfer. This design connects the EDA to evaluation: one test asks how the detector performs on familiar mixed-domain data, and the other asks what happens when it travels to a country excluded from training.

# Member 2: a controlled champion–challenger experiment

| Controlled factor | YOLO11n | YOLO11s |
|---|---:|---:|
| Role | Lightweight baseline | Capacity challenger |
| Parameters | 2.58 M | 9.41 M |
| Training manifest | Same hash-pinned 8,000 images | Same hash-pinned 8,000 images |
| Validation / test | Full 5,714 / 5,783 | Full 5,714 / 5,783 |
| Schedule | 30 epochs, 640 px, same seed/augmentation | Identical |

**Metric guide:** mAP@0.50 asks whether damage was found with reasonable overlap; mAP@0.50:0.95 rewards tight localization; recall measures missed damage; F1 balances precision and recall at a validation-selected confidence.

**Takeaway:** the comparison isolates model capacity—data composition, evaluation set, threshold-selection rule and timing hardware remain controlled.


For detection, we designed a controlled champion–challenger comparison between YOLO11n and YOLO11s. Both models use exactly the same hash-pinned 8,000-image training manifest, random seed, augmentation policy, 640-pixel input, and 30-epoch schedule. We still use the full validation and test sets, so the training budget does not shrink the final evaluation. Model selection uses the full validation set, and final reporting uses the untouched 5,783-image test set. The only main change is model capacity: YOLO11s is wider and deeper. This makes the comparison interpretable, because a performance difference is not caused by a different data sample or evaluation rule.

# Limited training compute, full-strength evaluation

| Stage | Images | Why it is used |
|---|---:|---|
| Full duplicate-safe training pool | 26,888 | Available supervised training data |
| Hash-pinned model-training subset | 8,000 | Same representative sample for both detectors |
| Full validation set | 5,714 | Checkpoint and confidence selection |
| Full shared test set | 5,783 | One locked final comparison |

The 8,000-image subset differs from the full training pool by at most **0.01 percentage points** across capture-domain and primary-class composition.

**Takeaway:** compute limits reduce the performance ceiling, but they do not weaken the fairness of the YOLO11n-versus-YOLO11s comparison.

Training both models for thirty epochs on all 26,888 training images exceeded the available Colab budget, so we reduced only the fitting set. Both detectors receive the exact same subset, seed, augmentation, image size, and schedule. More importantly, we do not shrink validation or test. The complete validation set still selects checkpoints and thresholds, and the complete test set still supports the final comparison. This design can tell us which architecture is better under the shared budget, although it does not claim the maximum accuracy possible with full-data training.

# Detection result: the larger model wins where it matters

<p align="center"><img src="artifacts/presentation/detection_model_comparison.png" alt="YOLO11n versus YOLO11s held-out detection metrics and compute tradeoff" width="980"></p>

| Shared test metric | YOLO11n | YOLO11s |
|---|---:|---:|
| mAP@0.50 | 0.4336 | **0.4439** |
| mAP@0.50:0.95 | 0.2033 | **0.2080** |
| Recall / F1 | 0.4268 / 0.4627 | **0.4447 / 0.4783** |
| D40 AP / recall | 0.2865 / 0.2621 | **0.3209 / 0.3024** |
| Single-image latency (L4) | 16.07 ms | 16.28 ms |

**Takeaway:** YOLO11s is the detection champion because it improves both overall detection and pothole recall with effectively unchanged single-image latency.


On the shared test set, YOLO11s wins every primary detection metric. Its mAP at fifty is 0.4439, compared with 0.4336 for YOLO11n. Its stricter mAP from fifty to ninety-five is also higher, so the gain is not limited to loose matches. Recall and F1 both improve, and the most meaningful class-level change is on D40 potholes: recall rises from 0.2621 to 0.3024, while D40 average precision rises from 0.2865 to 0.3209. The figure summarizes both accuracy and compute. The improvement is numerically modest, so we do not claim a dramatic breakthrough; we claim a consistent advantage on the locked test set, including the class we care most about. Single-image latency on the same L4 GPU is almost unchanged, although batched compute is higher. This balance of better D40 performance and nearly identical single-image latency is why the larger model earns the champion label.

# The champion gain is real—but training was compute-limited

<p align="center"><img src="artifacts/presentation/detection_learning_curves.png" alt="YOLO11n and YOLO11s learning curves across 30 epochs" width="900"></p>

| Paired bootstrap micro-F1 | YOLO11n | YOLO11s | Delta (95% CI) |
|---|---:|---:|---:|
| Overall | 0.4636 | 0.4830 | **+0.0194** [+0.0112, +0.0268] |
| D40 | 0.3215 | 0.3630 | **+0.0416** [+0.0158, +0.0688] |

Both primary runs achieve their best validation score at epoch 30.

**Takeaway:** the YOLO11s gain is statistically distinguishable from zero, but both curves suggest the compute-limited training schedule had not fully saturated.


We also checked whether this improvement could be sampling noise. A paired bootstrap compares both models on the same test images. The overall micro-F1 gain is about 0.019, with a ninety-five-percent confidence interval from 0.011 to 0.027. For D40, the gain is larger, about 0.042, and its confidence interval also remains above zero. This supports a real champion decision. However, both models reach their best validation score at epoch thirty, the final epoch, so the curves suggest that the compute-limited schedule had not fully saturated. The learning curves support the result while also motivating a longer future training study.

# Generalization test: unseen geography costs about one sixth of mAP

| Training geography | Strict US slice | Matched non-US slice |
|---|---:|---:|
| All-country 8k | **0.5064** | 0.4247 |
| Non-US 8k | 0.4206 | **0.4391** |

<p align="center"><img src="artifacts/member2/runs/figures/B8_miss_rate_slices.png" alt="Detection miss rates by class, object size, country, blur, and brightness" width="900"></p>

- Removing US training data reduces strict US mAP@0.50 by **16.9% relative**.
- Small-tercile boxes are missed **74.0%** of the time even by YOLO11s.
- D40 remains the worst class: **71.7% miss rate** at the selected operating point.

**Takeaway:** object size is the dominant failure driver, and geography adds a second measurable risk.


Average test performance does not guarantee geographic transfer. To test that, we retrained YOLO11s without US training images and evaluated a strict US slice. The all-country model reaches 0.5064 mAP at fifty on that slice, while the non-US model falls to 0.4206. That is a 16.9 percent relative loss. Notice that the non-US model is slightly stronger on the matched non-US slice. That does not cancel the US drop; it shows specialization and reminds us that geographic averages can hide which populations a model serves best. The miss-rate analysis points to an even broader limitation: YOLO11s misses 74 percent of small-tercile targets, and D40 has the worst class miss rate. So size is the dominant failure factor, while unseen geography adds a second measurable risk.

# Error analysis: localization and missed small damage dominate

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Dense multi-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Small-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 31%">
</div>
<p align="center"><small>Selected views from the five-case qualitative panel; the complete panel remains in the master notebook.</small></p>

| False-positive type (YOLO11s) | Share | What it means |
|---|---:|---|
| Localization | 43.8% | Correct damage, loose box |
| Background | 33.1% | Shadows, seams, markings, repairs |
| Duplicate | 18.4% | Extra box on an already found target |
| Wrong class | 4.7% | Taxonomy confusion is uncommon |

**Takeaway:** the bottleneck is not learning the four names; it is seeing tiny damage and drawing one tight box around it.


The qualitative examples explain what the aggregate metrics hide. YOLO11s often finds the correct damage category but places a box too loosely, so localization errors make up 43.8 percent of its false positives. Background confusion contributes another third, especially around shadows, seams, lane markings, and repaired pavement. Duplicate detections are also visible, while true wrong-class errors are relatively uncommon. The selected views pair ground truth with predictions, so we can see that a correct class label may still produce weak spatial evidence for an inspector. This means the model has mostly learned the four-class vocabulary. Its harder problem is detecting tiny, low-contrast damage and producing exactly one tight box around each target.

# Member 3: segmentation begins with a sparse-foreground problem

<table><tr>
<td width="50%"><img src="artifacts/member3/eda/m3_fig2_foreground_imbalance.png" alt="Pothole foreground imbalance in Pothole Mix" width="100%"></td>
<td width="50%"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned road images and pothole masks" width="100%"></td>
</tr></table>

- All **4,340** image-mask pairs decode correctly; no dimension mismatch.
- Only **1,184 images (27.3%)** contain pothole pixels.
- Among positive images, the median pothole covers only **2.58%** of the frame.
- Green crack pixels are background for this binary pothole task, creating valid hard negatives.

**Takeaway:** pixel accuracy would look strong by predicting mostly road, so model selection must focus on pothole IoU, Dice, recall and boundaries.


The segmentation task begins with a different imbalance. All 4,340 Pothole Mix image-mask pairs decode correctly, but only 1,184 images contain any pothole pixels. Even among positive images, the median pothole covers only 2.58 percent of the frame. The paired images and masks make this tangible: the annotation may occupy only a thin region inside a much larger road scene. Many remaining examples contain cracks or road texture without a pothole, so they are useful hard negatives. This makes ordinary pixel accuracy misleading: a model could label almost everything as background and still appear strong. We therefore focus on pothole IoU, Dice, recall, and boundary quality.

# Two segmentation models, one shared evaluation stack

| | DeepLabV3–MobileNetV3 | SegFormer-B0 |
|---|---|---|
| Core idea | Atrous convolution + multi-scale CNN context | Hierarchical transformer + lightweight MLP decoder |
| Expected strength | Local detail, recall, lower activation memory | Global/multi-scale context, compact parameter count |
| Main risk | Boundary loss and limited long-range context | Upsampling boundaries and lower recall |

**Shared protocol:** official 3,340 / 496 / 504 split, 512×512 input, paired augmentation, cross-entropy + soft-Dice loss, 40 epochs, validation-selected checkpoint and threshold.

**Metric guide:** IoU penalizes both missing and extra mask area; Dice summarizes overlap; recall measures missed pothole pixels; Boundary F1 tests whether the predicted edge follows the true edge.

**Takeaway:** architecture changes, but data, loss, evaluation code and test policy stay fixed.


We compare two segmentation families under one shared protocol. DeepLabV3 with MobileNetV3 uses atrous convolution and multi-scale CNN context. SegFormer-B0 uses a hierarchical transformer encoder and a lightweight decoder. Both receive the official train, validation, and test split, the same 512-by-512 input, paired spatial augmentation, combined cross-entropy and soft-Dice loss, and forty training epochs. We also use the same metric implementation and validation-selected checkpoint policy. This means IoU, Dice, recall, and Boundary F1 have exactly the same definition for both models. The experiment therefore stays focused on architecture rather than hidden pipeline differences.

# Segmentation result: a narrow win with a meaningful tradeoff

<table><tr>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig7_training_curves.png" alt="DeepLabV3 and SegFormer training curves" width="100%"></td>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig8_confusion_matrices.png" alt="Segmentation confusion matrices" width="100%"></td>
</tr></table>

| Test metric | DeepLabV3 | SegFormer-B0 |
|---|---:|---:|
| Pothole IoU / Dice | 0.6533 / 0.7903 | **0.6646 / 0.7985** |
| Precision / recall | 0.7889 / **0.7917** | **0.8623** / 0.7436 |
| Boundary F1 | 0.7188 | **0.7334** |
| Parameters | 11.02 M | **3.71 M** |

**Takeaway:** SegFormer wins overlap, precision and boundary quality with 3× fewer parameters; DeepLabV3 remains the recall-oriented alternative.


SegFormer-B0 is the overall segmentation champion, but the margin is narrow and the tradeoff matters. Its pothole IoU is 0.6646 and its Dice score is 0.7985, both slightly above DeepLabV3. It also has better precision and boundary F1, while using about one third as many parameters. DeepLabV3, however, has higher pothole recall: 0.7917 compared with 0.7436. The confusion matrices support this interpretation: SegFormer is more conservative and precise, whereas DeepLabV3 converts more true pothole pixels at the cost of additional false positives. So SegFormer is preferable for cleaner overlap, boundaries, and model size, but DeepLabV3 remains a reasonable option when missing a pothole is more costly than raising an extra alert.

# Look beyond the mean: best and worst segmentation cases

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Strong pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Difficult pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 100%">
</div>
<p align="center"><small>Selected strong and difficult cases; the complete comparison panel remains in the master notebook.</small></p>

Close potholes are segmented well (IoU roughly 0.88–0.92), while thin, shallow, distant damage can be missed completely. The per-image IoU distribution is therefore bimodal: a single mean hides a genuine failure cluster.

**Takeaway:** the champion is not “solved”—qualitative failures explain why boundary metrics and human review remain necessary.


The best and worst cases show why one mean score is not enough. In close, clearly visible potholes, both models can reach image-level IoU near 0.9 and follow the damaged region well. In contrast, thin, shallow, distant, or low-contrast potholes may be missed completely. This creates a real failure cluster rather than a smooth decline in quality. The examples also show that a good interior overlap does not always guarantee an accurate boundary. That is why we report boundary F1 and preserve human review instead of treating the champion as a solved segmentation system.

# From two models to one human-review application

<table><tr>
<td width="50%"><img src="artifacts/member3/app/CPU-2.png" alt="Verified CPU Gradio road damage analysis" width="100%"></td>
<td width="50%"><img src="artifacts/member3/app/GPU.png" alt="Verified GPU Gradio road damage analysis" width="100%"></td>
</tr></table>

The app shows detection boxes, the segmentation overlay, confidences, damaged-area percentage, model/runtime metadata and a transparent Low/Medium/High prototype priority with reasons.

The captured CUDA run completed the full YOLO11s + SegFormer path in **56 ms**; CPU first-call latency exceeded one second. These app timings include end-to-end overhead and are separate from controlled per-model benchmarks.

**Takeaway:** the interface exposes evidence and limitations; its priority is a triage suggestion, not a civil-engineering rating.


We combine the two champions in a Gradio review application. The interface displays YOLO boxes and class confidences, the SegFormer mask and damaged-area percentage, model and runtime metadata, and a transparent Low, Medium, or High prototype priority with explicit reasons. The screenshots demonstrate both CPU and GPU behavior using the same visible outputs and review boundary. In the captured CUDA run, the complete two-model path took 56 milliseconds; CPU first-call latency was above one second. These are end-to-end application timings, not the controlled per-model benchmark. Most importantly, the priority is only a triage suggestion, not a certified pavement-condition or repair rating.

# Final scorecard: champion does not mean universally best

<p align="center"><img src="artifacts/final/model_tradeoff_summary.svg" alt="Detection and segmentation champion-challenger tradeoff summary" width="1100"></p>

**Takeaway:** both champions win narrowly. YOLO11s earns deployment preference through significant pothole-recall gains; SegFormer wins overlap and size, but DeepLabV3 is defensible when missed potholes cost more than false alarms.


This scorecard is our combined model decision. YOLO11s is selected because its statistically supported improvement includes better pothole recall at nearly the same single-image latency. SegFormer-B0 is selected because it gives the best overlap and boundary quality with a much smaller parameter count. Neither choice wins every metric. That is intentional: a champion should be selected against the deployment objective, while the alternative remains documented for situations where recall, compute, or false alarms have a different cost.

# Deployment must be observable, versioned, and reversible

<p align="center"><img src="artifacts/final/deployment_architecture.svg" alt="Versioned road damage deployment architecture with monitoring and rollback" width="1080"></p>

**Operational loop:** monitor → label reviewed failures → train a versioned challenger → gate on locked tests → canary → promote or roll back.

**Takeaway:** model operations preserve the same principle as our experiment design: version everything, protect the test set, measure drift and keep a previous champion ready.


A useful model also needs an operational safety loop. We would version the data, preprocessing, models, thresholds, and review policy together; monitor input drift, confidence, latency, and reviewed failures; and train a challenger only from labeled evidence. A locked test gate and canary deployment protect the current champion, while rollback keeps the previous version available. This makes future updates observable and reversible instead of silently replacing one model with another.

# The next experiments should target observed failures

| Priority | Next experiment | Evidence of improvement |
|---|---|---|
| Small damage | Higher-resolution crops or tiled detection | Better small-tercile and D40 recall |
| Geographic transfer | Add labeled countries and camera types | Smaller held-out-domain performance drop |
| Training ceiling | Longer full-data schedules | Validation curves that clearly plateau |
| Reliability | Confidence calibration and review study | Better agreement between confidence and correctness |
| Pothole boundaries | Boundary-aware loss or refinement | Higher Boundary F1 without recall collapse |
| Environmental coverage | Night, rain, snow, glare, and water tests | Documented acceptance ranges and failure triggers |

**Takeaway:** future work is prioritized by measured errors, not by adding a more complicated model without a clear failure target.

The current evidence gives us a practical research order. Small-object detection comes first because it is the largest measured source of misses. Broader geographic and environmental data comes next because the US holdout already demonstrates domain sensitivity. Longer training can test whether both YOLO curves continue improving, while calibration and a reviewer study can determine whether displayed confidence is genuinely useful to people. For segmentation, boundary-aware methods should be judged against both edge quality and recall so that cleaner masks do not simply become more conservative.

# Conclusion: decision support, not autonomous maintenance

### What remains hard

- **Small-object recall:** YOLO11s still misses 74% of small-tercile boxes; D40 recall is 0.3024.
- **Domain shift:** removing US training data costs 16.9% relative mAP on strict US images.
- **Localization and boundaries:** detection mAP@0.50:0.95 is about 0.21; segmentation Boundary F1 is about 0.73.
- **Coverage:** night, snow, water, glare, unusual cameras and new jurisdictions are not acceptance-tested.

### Final answer to the opening questions

- **What broke and where?** YOLO11s provides four-class boxes.
- **How much area?** SegFormer-B0 provides the pothole mask and footprint.
- **Who decides?** A human inspector, with evidence and explicit uncertainty.

> **Closing line:** Our strongest result is not automation—it is an auditable way to prioritize road images without hiding where the models fail.

# Questions?


To conclude, our system answers the opening questions with visible evidence: YOLO11s identifies what broke and where, while SegFormer-B0 estimates the pothole footprint. But the limitations are just as important. Small objects, unfamiliar geography, tight localization, and difficult boundaries remain unsolved, and several real road conditions are not yet acceptance-tested. Our final claim is therefore modest and practical: the project offers an auditable way to prioritize road images, while a human inspector keeps authority over the final maintenance decision.

# Appendix — team roles and sources

| Member | Presentation ownership |
|---|---|
| **Member 1 — Austin Wang** | Problem framing, RDD2022 data engineering, EDA, split design, and combined scorecard |
| **Member 2 — Kevin Fan** | Detection experiment, results, generalization, error analysis, and deployment operations |
| **Member 3 — RJ Xia** | Segmentation data, models, results, qualitative failures, application, and conclusion |

## Key sources

1. Arya, D. et al. RDD2022 dataset and paper: https://doi.org/10.6084/m9.figshare.21431547 and https://doi.org/10.1002/gdj3.260 (CC BY 4.0).
2. Pothole Mix v1.0: https://doi.org/10.17632/kfth5g2xk3.2; component-license record in `artifacts/member3/pothole_mix_provenance.json`.
3. Ultralytics YOLO11: https://docs.ultralytics.com/models/yolo11.
4. Chen, L.-C. et al., DeepLabV3: https://arxiv.org/abs/1706.05587; Howard, A. et al., MobileNetV3: https://arxiv.org/abs/1905.02244.
5. Xie, E. et al., SegFormer: https://arxiv.org/abs/2105.15203.
6. Gradio: https://www.gradio.app/docs.
7. Opening photograph: Prosthetic Head, “Marked Pothole,” Wikimedia Commons, CC BY-SA 4.0: https://commons.wikimedia.org/wiki/File:Marked_Pothole.jpg.

Exact methods, complete tables, code, saved outputs and the full reference list remain in `Road_Damage_Final_Project_Master_New.ipynb`.
